```markdown
# Step 1: Understanding BERT and XLM-RoBERTa

### BERT (Bidirectional Encoder Representations from Transformers)
* **Architecture**: Based on the Transformer encoder architecture.
* **Training**: Pre-trained using Masked Language Modeling (MLM) and Next Sentence Prediction (NSP).
* **Language**: Primarily monolingual (e.g., English), though multilingual versions exist (mBERT).

### XLM-RoBERTa (Cross-lingual Language Model - Robustly Optimized BERT Approach)
* **Architecture**: An extension of RoBERTa trained on a massive multilingual dataset (CommonCrawl).
* **Training**: Uses only the Masked Language Modeling (MLM) objective, discarding NSP.
* **Language**: Excellent cross-lingual performance across over 100 languages.

### Tokenization
Both models process text by breaking down strings into subword tokens, mapping them to vocabulary indices (`input_ids`), and outputting masks to distinguish content from padding.
```

In [9]:
# Install transformers library if not already installed
!pip install -q transformers

```markdown
# Step 2: Tokenizing Text

Here we instantiate the tokenizers and experiment with single and pairwise text tokenization using `encode_plus` and `decode`.

In [10]:
from transformers import BertTokenizer, XLMRobertaTokenizer

# Instantiate tokenizers
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

sentence_1 = "Natural Language Processing is fascinating."
sentence_2 = "We can represent text as numbers."

print("--- BERT Single Sentence ---")
# Calling the tokenizer directly serves as the standard, robust encoding pipeline (equivalent to encode_plus)
bert_encoded = bert_tokenizer(
    sentence_1,
    add_special_tokens=True,
    return_token_type_ids=True,
    return_attention_mask=True
)
print("Input IDs:      ", bert_encoded['input_ids'])
print("Token Type IDs: ", bert_encoded['token_type_ids'])
print("Attention Mask: ", bert_encoded['attention_mask'])
print("Decoded:        ", bert_tokenizer.decode(bert_encoded['input_ids']))

print("\n--- XLM-RoBERTa Two Sentences ---")
xlm_encoded = xlm_tokenizer(
sentence_1,
sentence_2,
add_special_tokens=True
)
print("Input IDs:      ", xlm_encoded['input_ids'])
print("Decoded:        ", xlm_tokenizer.decode(xlm_encoded['input_ids']))

--- BERT Single Sentence ---
Input IDs:       [101, 3019, 2653, 6364, 2003, 17160, 1012, 102]
Token Type IDs:  [0, 0, 0, 0, 0, 0, 0, 0]
Attention Mask:  [1, 1, 1, 1, 1, 1, 1, 1]
Decoded:         [CLS] natural language processing is fascinating. [SEP]

--- XLM-RoBERTa Two Sentences ---
Input IDs:       [0, 24955, 83658, 123996, 214, 83, 102919, 1916, 5, 2, 2, 1401, 831, 33636, 7986, 237, 101935, 5, 2]
Decoded:         <s> Natural Language Processing is fascinating.</s></s> We can represent text as numbers.</s>


```markdown
# Step 3: Preparing Input Data for the Model

We ensure input sequences are of uniform length using padding and truncation, and check the vocab sizes and special token configurations.

In [11]:
max_len = 16

# Call the tokenizer directly instead of encode_plus to avoid lookup attribute errors
prepared_inputs = xlm_tokenizer(
    sentence_1,
    max_length=max_len,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    add_special_tokens=True
)

print("Input IDs:", prepared_inputs['input_ids'])
print("Attention Mask:", prepared_inputs['attention_mask'])
print("\nSpecial Tokens Map:", xlm_tokenizer.special_tokens_map)
print("Vocab Size:", xlm_tokenizer.vocab_size)

Input IDs: [0, 24955, 83658, 123996, 214, 83, 102919, 1916, 5, 2, 1, 1, 1, 1, 1, 1]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

Special Tokens Map: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}
Vocab Size: 250002


```markdown
# Step 4: Loading and Exploring the Dataset

We will read the sample dataset available in the runtime environment (`/content/sample_submission.csv` or standard housing data) to demonstrate data ingestion.

In [12]:
import pandas as pd

# Load the sample submission dataset
df = pd.read_csv('/content/sample_submission.csv')

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (5195, 2)


,id,prediction
0,c6d58c3f69,1
1,cefcc82292,1
2,e98005252c,1
3,58518c10ba,1
4,c32b0d16df,1


```markdown
# Step 5: Creating Cross-Validation Folds

Using `StratifiedKFold` to create 5 distinct splits of the data while maintaining class balance.

In [13]:
import numpy as np
from sklearn.model_selection import StratifiedKFold

# Dummy targets representing class labels for demonstration
# Let's say we classify based on a threshold on the sample submission target
np.random.seed(42)
df['label'] = np.random.choice([0, 1], size=len(df))

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

folds = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X=df, y=df['label'])):
    train_split = df.iloc[train_idx]
    val_split = df.iloc[val_idx]
    folds.append((train_split, val_split))
    print(f"Fold {fold + 1} | Train size: {len(train_split)} | Val size: {len(val_split)} | Val Class Ratio: {val_split['label'].mean():.2f}")

Fold 1 | Train size: 4156 | Val size: 1039 | Val Class Ratio: 0.50
Fold 2 | Train size: 4156 | Val size: 1039 | Val Class Ratio: 0.50
Fold 3 | Train size: 4156 | Val size: 1039 | Val Class Ratio: 0.50
Fold 4 | Train size: 4156 | Val size: 1039 | Val Class Ratio: 0.50
Fold 5 | Train size: 4156 | Val size: 1039 | Val Class Ratio: 0.50
